# Task 3 - Pairwise Feature Engineering and Analysis

This notebook reviews the saved training-only Task 3 artifacts. It does not train a classifier, tune a threshold, access test data, or generate predictions.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path.cwd()
if not (ROOT / 'outputs' / 'task3_outputs').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / '.task3_vendor'))
import pyarrow.parquet as pq

OUT = ROOT / 'outputs' / 'task3_outputs'
assert OUT.exists(), f'Missing Task 3 outputs: {OUT}'

In [ ]:
pair_counts = pd.read_csv(OUT / 'task3_pair_counts.csv')
split_summary = pd.read_csv(OUT / 'task3_split_summary.csv')
feature_schema = pd.read_csv(OUT / 'task3_feature_schema.csv')
feature_stats = pd.read_csv(OUT / 'task3_feature_statistics.csv')
redundancy = pd.read_csv(OUT / 'task3_feature_redundancy.csv')
source_analysis = pd.read_csv(OUT / 'task3_source_feature_analysis.csv')
group_analysis = pd.read_csv(OUT / 'task3_match_group_feature_analysis.csv')
recovery_analysis = pd.read_csv(OUT / 'task3_recovered_positive_analysis.csv')
recovery_counts = pd.read_csv(OUT / 'task3_recovery_group_counts.csv')
hard_negatives = pd.read_csv(OUT / 'task3_hard_negative_sample.csv')
difficult_positives = pd.read_csv(OUT / 'task3_difficult_positive_sample.csv')
validation = pd.read_csv(OUT / 'task3_validation_report.csv')
leakage = pd.read_csv(OUT / 'task3_leakage_audit.csv')
runtime = pd.read_csv(OUT / 'task3_runtime_memory.csv')
manifest = json.loads((OUT / 'task3_run_manifest.json').read_text())

## Frozen candidate parity and pair balance

In [ ]:
display(pair_counts)
display(pd.Series(manifest['frozen_metrics'], name='value').to_frame())

## Typed pair dataset

In [ ]:
pair_file = pq.ParquetFile(OUT / 'task3_pair_features.parquet')
print(f'Rows: {pair_file.metadata.num_rows:,}')
print(f'Columns: {pair_file.metadata.num_columns}')
print(f'Row groups: {pair_file.num_row_groups}')
display(pair_file.read_row_group(0).slice(0, 5).to_pandas())

## Feature schema and compact Task 4 recommendation

In [ ]:
print('All columns:', len(feature_schema))
print('Recommended Task 4 features:', int(feature_schema['recommended_task4'].sum()))
display(feature_schema.groupby(['role', 'family']).size().rename('columns').reset_index())
display(feature_schema[feature_schema['recommended_task4']][['column', 'family', 'requires_fitted_state']])

## Positive versus hard-negative evidence

The statistics use every positive pair plus a deterministic sample of candidate negatives. Univariate AUC is descriptive only and is not classifier importance.

In [ ]:
display(feature_stats.head(25))
display(feature_stats.sort_values(['directionless_univariate_auc', 'feature']).head(20))

## Redundancy review

In [ ]:
display(redundancy)
print('Jaro was removed after the pilot because it correlated 0.998-0.999 with Jaro-Winkler.')
print('A separate pairwise TF-IDF was not fit; frozen retrieval scores retain that evidence without introducing Task 3 fitted state.')

## S2 versus S3 and match-count groups

In [ ]:
core = [
    'name_token_set', 'address_token_set', 'address_token_jaccard',
    'address_number_jaccard', 'retrieval_best_rank',
    'retrieval_signal_count_top100', 'independent_strong_signal_count',
]
source_positive = source_analysis[source_analysis['label'].eq(1) & source_analysis['feature'].isin(core)]
display(source_positive.pivot(index='feature', columns='candidate_source', values='mean'))
group_positive = group_analysis[group_analysis['label'].eq(1) & group_analysis['feature'].isin(core)]
display(group_positive.pivot(index='feature', columns='match_group', values='mean')[['1', '2', '3-5', '6+']])

## Task 2 baseline, Task 2.5 recoveries, and remaining misses

In [ ]:
display(recovery_counts)
recovery_core = recovery_analysis[recovery_analysis['feature'].isin(core + [
    'transliteration_levenshtein_gain', 'address_conflicting_number_count'
])]
display(recovery_core.pivot(index='feature', columns='positive_group', values='mean'))

## Hard cases

In [ ]:
sample_columns = [
    'sample_reason', 'source1_entity_id', 'candidate_entity_id',
    's1_name', 'candidate_name', 's1_address', 'candidate_address',
    'name_token_set', 'address_token_set', 'address_number_jaccard',
    'retrieval_signal_count_top100', 'retrieval_best_rank',
]
display(hard_negatives[sample_columns].head(30))
display(difficult_positives[sample_columns].head(30))

## Entity split, leakage audit, and validation

In [ ]:
display(split_summary)
display(leakage)
display(validation)
assert leakage['status'].eq('PASS').all()
assert validation['status'].eq('PASS').all()

## Runtime and storage

In [ ]:
display(runtime)
print(f"Total runtime: {manifest['total_runtime_seconds'] / 60:.2f} minutes")
print(f"Peak RSS: {manifest['peak_rss_mb']:.1f} MB")
print(f"Pair dataset: {manifest['pair_dataset_mb']:.1f} MB")

## Task 4 recommendation

Start with a class-weighted gradient-boosted tree on the frozen entity split using the 66 recommended features. Keep regularized logistic regression as a linear benchmark. Evaluate ranking and entity-level outcomes on validation before any threshold selection.

In [ ]:
for key in [
    'classifier_work_performed', 'threshold_tuning_performed',
    'probability_calibration_performed', 'test_data_used',
    'test_predictions_created', 'submission_created',
]:
    assert manifest[key] is False
print('Boundary check passed: Task 3 contains features and analysis only.')

## Optional reproducibility run

The saved artifacts are sufficient for review. Enable the flag only for an intentional full rebuild.

In [ ]:
RUN_FULL_TASK3 = False
if RUN_FULL_TASK3:
    env = dict(__import__('os').environ)
    env['PYTHONPATH'] = str(ROOT / '.task3_vendor')
    subprocess.run(
        ['python3', str(ROOT / 'src' / 'task3_pairwise_features.py')],
        cwd=ROOT, env=env, check=True,
    )